```bash
gcloud storage cp -r gs://grouping-data/runs/./2025-12-19-15-53-06-gte-output/training gte-finetuned/

gcloud storage cp -r gs://grouping-data/final_csvs .
```

from https://console.cloud.google.com/storage/browser/grouping-data/final_csvs

In [1]:
import textwrap

import polars as pl
from sentence_transformers.util import pairwise_cos_sim

import grouping_trainer as gt

import utils

In [3]:
model = gt.utils.SentenceTransformer("gte-finetuned/training", trust_remote_code=True)

In [ ]:
df = utils.load_val_df(sample_size=1_000)

In [ ]:
X = model.encode(
    df["query_stacktrace_string"].to_list() + df["candidate_stacktrace_string"].to_list(),
    show_progress_bar=True,
    batch_size=2,
)

Batches:   0%|          | 0/939 [00:00<?, ?it/s]

In [5]:
assert X.shape == (len(df) * 2, 768)

In [6]:
Q, C = X[: len(df)], X[len(df) :]

In [7]:
S = pairwise_cos_sim(Q, C).detach().numpy()

In [18]:
def record_to_prompt(record: pl.DataFrame) -> str:
    return textwrap.dedent(
        """\
        {query_stacktrace_string}
        {delimiter}
        {candidate_stacktrace_string}
        {delimiter}
        Similarity: {similarity:.3f}"""
    ).format(
        query_stacktrace_string=record["query_stacktrace_string"],
        delimiter="\n\n" + "-" * 100 + "\n\n",
        candidate_stacktrace_string=record["candidate_stacktrace_string"],
        similarity=record["similarity"],
    )

In [ ]:
df_least_similar = df.with_columns(similarity=S).filter(pl.col("similarity") > 0.70).sort(by="similarity")[:50]

In [ ]:
for record in df_least_similar.rows(named=True):
    print(record_to_prompt(record))
    print("\n\n" + "#" * 100 + "\n\n")